In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv("/kaggle/input/q3-ka-ai-2026/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
for col in df.columns:
  df[col].fillna(df[col].mode()[0])

In [ ]:
# Task 2: Write your code here:
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows.")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")



In [ ]:
# Task 3: Write your code here:

# No encoding needed, all features are non-categorical variables

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df


In [ ]:
# Task 5: Write your code here:
import seaborn as sns
import matplotlib.pyplot as plt
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

# Target is imbalanced!

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split
X = df.drop("Target", axis=1).astype(float)
y = df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
import numpy as np

n_splits = 5 # K
mae_scores = []
lr_accuracy = []
lr_f1 = []

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model = CatBoostClassifier()
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  # Store results
  lr_accuracy.append(accuracy)
  lr_f1.append(f1)

  mae_scores.append(mean_absolute_error(y_test, y_pred))
print()
print(f"Accuracy: {np.mean(lr_accuracy)}")
print(f"Accuracy: {np.mean(lr_f1)}")
print(f"Accuracy: {np.mean(mae_scores)}")

In [ ]:
# Task 1: Write your code here:
importances = {}

importances['CatBoost Classifier'] = model.feature_importances_
fig, axes = plt.subplots(1, 3, figsize=(18, 20))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.show()

In [ ]:
# Task 2: Write your code here:

#Golden_Feature: P_2

print(df['P_2'].describe())

In [ ]:
# Task Bonus: Write your code here: